In [1]:
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
def visualize_value_change(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    sensitive_col: str = "DP_RACE",
    value_col: str = "Value",
    font_size: int = 26,
    ticks_font_size: int = 20,
    title: str = "",
    y_label: str = "",
    initial_state: str = "",
    jitter_amount: float = 0.1,
    legend_filename: str = "value_change_legend.pdf",
    save_path: str | None = None,
    custom_labels: dict | None = None,
) -> plt.Figure:
    """
    Visualizes the change in a specified sensitive column across different value
    categories between two dataframes with jittered dots and a separate legend file.
    Arrows are drawn only if the 'Value' is different between the two connected states.
    """
    # Merge the two dataframes based on the common columns
    merged_df = df1.merge(df2, on="dataset", suffixes=("_df1", "_df2"))

    fig, _ax = plt.subplots(figsize=(12, 5))

    # Styling for the initial state scatter plot
    marker_face_color = "#004D40"
    marker_edge_color = "black"
    marker_size = 200
    marker_linewidth = 1.2

    # Arrow styling
    arrow_head_width = 0.05
    arrow_head_length = 0.05
    arrow_alpha = 0.7
    arrow_linewidth = 1.5
    arrow_color = "gray"

    # Store jittered positions: {dataset_id: (x, y)}
    pos_map_df1 = {}
    pos_map_df2 = {}

    rng = np.random.default_rng()

    # Plot the initial state with jitter
    unique_vals = sorted(df1[value_col].unique())
    for val in unique_vals:
        subset = df1[df1[value_col] == val]
        y_vals = subset[sensitive_col].astype(float)
        x_base = int(float(val))
        jitter = rng.uniform(-jitter_amount, jitter_amount, len(y_vals))
        x_vals = [x_base + j for j in jitter]

        # Store positions
        for i, dataset_id in enumerate(subset["dataset"]):
            pos_map_df1[dataset_id] = (x_vals[i], y_vals.to_numpy()[i])

        plt.scatter(
            x_vals,
            y_vals,
            facecolors=marker_face_color,
            edgecolors=marker_edge_color,
            marker="o",
            s=marker_size,
            linewidths=marker_linewidth,
            label=initial_state if val == df1[value_col].unique()[0] else "",  # Label only once
            zorder=2,  # Ensure initial state points are on top of arrows
            alpha=0.6,
        )

    # Plot the final state points with jitter
    for val in sorted(df2[value_col].unique()):
        subset = df2[df2[value_col] == val]
        y_vals = subset[sensitive_col].astype(float)
        x_base = int(float(val))
        jitter = rng.uniform(-jitter_amount, jitter_amount, len(y_vals))
        x_vals = [x_base + j for j in jitter]

        # Store positions
        for i, dataset_id in enumerate(subset["dataset"]):
            pos_map_df2[dataset_id] = (x_vals[i], y_vals.to_numpy()[i])

        plt.scatter(
            x_vals,
            y_vals,
            s=200,
            color="#FFC107",
            edgecolor="black",
            label="FedAVG" if val == df2[value_col].unique()[0] else "",  # Label only once
            zorder=2,  # Ensure final state points are on top of arrows
            alpha=0.8,
        )

    # Draw arrows based on the 'dataset' identifier
    for _index, row in merged_df.iterrows():
        dataset_id = row["dataset"]

        if dataset_id in pos_map_df1 and dataset_id in pos_map_df2:
            initial_x, initial_y = pos_map_df1[dataset_id]
            final_x, final_y = pos_map_df2[dataset_id]

            # Only draw arrow if there is a significant change in position
            # Magic values 0.01 and 0.001 are kept for now but could be constants.
            x_threshold = 0.01
            y_threshold = 0.001
            if abs(initial_x - final_x) > x_threshold or abs(initial_y - final_y) > y_threshold:
                plt.arrow(
                    initial_x,
                    initial_y,
                    final_x - initial_x,
                    final_y - initial_y,
                    head_width=arrow_head_width,
                    head_length=arrow_head_length,
                    fc=arrow_color,
                    ec=arrow_color,
                    alpha=arrow_alpha,
                    linewidth=arrow_linewidth,
                    length_includes_head=True,
                    zorder=1,
                )

    # Customize the x-axis ticks
    if custom_labels:
        ticks = sorted(custom_labels.keys())
        labels = [custom_labels[t] for t in ticks]
        plt.xticks(ticks=ticks, labels=labels, fontsize=ticks_font_size)
    else:
        plt.xticks(ticks=unique_vals, labels=[str(int(float(v))) for v in unique_vals], fontsize=ticks_font_size)

    plt.yticks(fontsize=ticks_font_size)
    plt.xlabel("Sensitive Group Value", fontsize=font_size)
    plt.ylabel(y_label, fontsize=font_size)
    plt.title(title, fontsize=font_size)
    plt.grid(visible=True)

    # Create a separate figure for the legend
    if save_path:
        fig_legend = plt.figure(figsize=(6, 1))
        ax_legend = fig_legend.add_subplot(111)
        initial_patch = mpatches.Patch(facecolor="#004D40", edgecolor="black", alpha=0.6, label=initial_state)
        # Better label handling needed, but for now:
        fedavg_patch = mpatches.Patch(facecolor="#FFC107", edgecolor="black", alpha=0.8, label="FedAVG")

        ax_legend.legend(handles=[initial_patch, fedavg_patch], fontsize=20, loc="center", frameon=False, ncol=2)
        ax_legend.axis("off")

        legend_path = Path(save_path).parent / legend_filename
        fig_legend.tight_layout()
        fig_legend.savefig(legend_path)
        plt.close(fig_legend)

        plt.tight_layout()
        plt.savefig(save_path, bbox_inches="tight", dpi=150)
        plt.close(fig)
        return fig
    plt.tight_layout()
    plt.show()
    return fig